# VQ-VAE v10 Training - Auxiliary Volume Head

## Overview
v10 takes a fundamentally different approach to fixing the volume ratio problem.

## The Problem (v8-v9.1)
Previous approaches tried to fix volume through loss penalties on **discrete predictions (argmax)**:
- Volume penalty acts on argmax, so gradient signal is sparse
- By epoch 3-4, most voxels have confident predictions → ~0 gradient for volume
- Air boost overcorrects, causing 40% structure erasure

## The v10 Solution: Auxiliary Volume Head
Instead of fighting the decoder with loss weights, we add a **separate head that predicts volume directly from the latent**.

### Key Innovation
```python
# Volume is now a REGRESSION task, not classification
volume_pred = self.volume_head(z_q)  # Predicts normalized volume [0-1]
volume_loss = F.l1_loss(volume_pred, gt_volume_normalized)
```

### Why This Works
1. **Clean gradient flow**: Volume head operates on `z_q`, not through softmax
2. **Regression, not classification**: Continuous gradients instead of sparse argmax
3. **Separate supervision**: Volume has its own explicit target
4. **No air boost**: Simple class weights instead of aggressive boosting

## Architecture Changes
- **VolumeHead**: MLP that predicts normalized volume from pooled latent
- **No two-phase training**: Single phase, volume head provides clean signal throughout
- **No frequency weighting**: Removed perverse incentive for over-prediction
- **No air_boost**: Replaced with simple class weights for air tokens

## Expected Results
| Metric | v9.1 (current) | v10 Target |
|--------|----------------|------------|
| Volume Ratio | 0.6x | 0.9-1.1x |
| Recall | 60% | 85%+ |
| False Air Rate | 40-55% | <10% |
| Building Accuracy | 45% | 45-55% |

## 1. Setup - Mount Google Drive

**Technical:** Mounts Google Drive filesystem to access training data and save model checkpoints.

**Simple:** Connects to your Google Drive to load Minecraft structure data and save the trained model.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

DRIVE_BASE = '/content/drive/MyDrive/minecraft_ai'
OUTPUT_DIR = '/content/drive/MyDrive/minecraft_ai/vqvae_v10'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

## 2. Imports

**Technical:** Imports PyTorch for deep learning, numpy for numerical operations, h5py for HDF5 files, matplotlib for plotting.

**Simple:** Loads all required software libraries.

In [ ]:
import json
import random
import time
from pathlib import Path
from typing import Dict, List, Tuple, Set, Optional
from collections import defaultdict

import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 3. RFSQ Quantization Module

**Technical:** Implements Residual Finite Scalar Quantization (RFSQ) with LayerNorm conditioning. Discretizes continuous latent vectors into finite codes using straight-through estimator for gradient flow.

**Simple:** Converts the AI's internal representation into discrete codes (analog to digital). This compression forces learning the most important features.

In [ ]:
class FSQ(nn.Module):
    """Finite Scalar Quantization - quantizes each dimension to fixed levels."""
    def __init__(self, levels: List[int], eps: float = 1e-3):
        super().__init__()
        self.levels = levels
        self.dim = len(levels)
        self.eps = eps
        self.codebook_size = int(np.prod(levels))
        self.register_buffer('_levels', torch.tensor(levels, dtype=torch.float32))
        basis = []
        acc = 1
        for L in reversed(levels):
            basis.append(acc)
            acc *= L
        self.register_buffer('_basis', torch.tensor(list(reversed(basis)), dtype=torch.long))
        half_levels = [(L - 1) / 2 for L in levels]
        self.register_buffer('_half_levels', torch.tensor(half_levels, dtype=torch.float32))

    def forward(self, z: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        z_bounded = torch.tanh(z)
        z_q = self._quantize(z_bounded)
        z_q = z_bounded + (z_q - z_bounded).detach()  # Straight-through
        indices = self._to_indices(z_q)
        return z_q, indices

    def _quantize(self, z: torch.Tensor) -> torch.Tensor:
        z_q_list = []
        for i in range(self.dim):
            half_L = self._half_levels[i]
            z_i = z[..., i] * half_L
            z_i = torch.round(z_i).clamp(-half_L, half_L) / half_L
            z_q_list.append(z_i)
        return torch.stack(z_q_list, dim=-1)

    def _to_indices(self, z_q: torch.Tensor) -> torch.Tensor:
        indices = torch.zeros(z_q.shape[:-1], dtype=torch.long, device=z_q.device)
        for i in range(self.dim):
            L = self._levels[i].long()
            half_L = self._half_levels[i]
            level_idx = ((z_q[..., i] * half_L) + half_L).round().long().clamp(0, L - 1)
            indices = indices + level_idx * self._basis[i]
        return indices

    def get_codebook_usage(self, indices: torch.Tensor) -> Tuple[float, float]:
        flat = indices.flatten()
        counts = torch.bincount(flat, minlength=self.codebook_size).float()
        usage = (counts > 0).float().mean().item()
        probs = counts / counts.sum()
        probs = probs[probs > 0]
        entropy = -(probs * torch.log(probs)).sum()
        perplexity = torch.exp(entropy).item()
        return usage, perplexity


class InvertibleLayerNorm(nn.Module):
    """LayerNorm that stores statistics for inverse transformation."""
    def __init__(self, num_features: int, eps: float = 1e-5):
        super().__init__()
        self.num_features = num_features
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(num_features))
        self.bias = nn.Parameter(torch.zeros(num_features))
        self.register_buffer('stored_mean', None, persistent=False)
        self.register_buffer('stored_std', None, persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        self.stored_mean = x.mean(dim=(1, 2, 3), keepdim=True)
        self.stored_std = x.std(dim=(1, 2, 3), keepdim=True) + self.eps
        x_norm = (x - self.stored_mean) / self.stored_std
        return x_norm * self.weight + self.bias

    def inverse(self, x_norm: torch.Tensor) -> torch.Tensor:
        x = (x_norm - self.bias) / self.weight
        return x * self.stored_std + self.stored_mean


class RFSQStage(nn.Module):
    """Single stage of RFSQ with LayerNorm conditioning."""
    def __init__(self, levels: List[int]):
        super().__init__()
        self.fsq = FSQ(levels)
        self.layernorm = InvertibleLayerNorm(len(levels))

    def forward(self, residual: torch.Tensor):
        z_norm = self.layernorm(residual)
        z_q_norm, indices = self.fsq(z_norm)
        z_q = self.layernorm.inverse(z_q_norm)
        new_residual = residual - z_q
        return z_q, new_residual, indices


class RFSQ(nn.Module):
    """Residual FSQ with multiple stages."""
    def __init__(self, levels_per_stage: List[int], num_stages: int = 2):
        super().__init__()
        self.num_stages = num_stages
        self.stages = nn.ModuleList([RFSQStage(levels_per_stage) for _ in range(num_stages)])
        self._usage_indices = []

    def reset_usage(self):
        self._usage_indices = []

    def forward(self, z: torch.Tensor):
        residual = z
        z_q_sum = torch.zeros_like(z)
        all_indices = []
        for stage in self.stages:
            z_q, residual, indices = stage(residual)
            z_q_sum = z_q_sum + z_q
            all_indices.append(indices)
        self._usage_indices.append(all_indices)
        return z_q_sum, all_indices

    def get_usage_stats(self) -> Dict:
        if not self._usage_indices:
            return {}
        stats = {}
        for stage_idx in range(self.num_stages):
            all_stage_indices = torch.cat([b[stage_idx].flatten() for b in self._usage_indices])
            usage, perplexity = self.stages[stage_idx].fsq.get_codebook_usage(all_stage_indices)
            stats[f'stage{stage_idx}'] = (usage, perplexity)
        return stats

print("RFSQ modules defined")

## 4. VolumeHead - Auxiliary Volume Prediction

**Technical:** A small MLP that predicts the normalized volume (structure voxels / total voxels) directly from the quantized latent. Uses adaptive pooling to reduce 8x8x8 latent to 4x4x4, then MLP to scalar.

**Simple:** A separate mini-network that looks at the compressed representation and predicts "how much of this structure is solid blocks vs air?". This gives clean gradient signal for volume control.

### Why This Is The Key Innovation
- Volume prediction is **regression** (continuous gradients)
- Operates on latent `z_q`, not through softmax
- Each structure gets one scalar target (clean supervision)
- Model can't hide volume bias in logit initialization

In [ ]:
class VolumeHead(nn.Module):
    """Predicts normalized volume from quantized latent representation.
    
    This is the key innovation in v10:
    - Volume becomes a regression task with clean gradients
    - Operates on z_q (latent), not on decoder logits
    - Provides explicit volume supervision
    """
    def __init__(self, latent_channels: int = 4, hidden_dim: int = 128):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool3d(4)  # 8x8x8 -> 4x4x4
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(4 * 4 * 4 * latent_channels, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()  # Output 0-1 (normalized volume)
        )

    def forward(self, z_q: torch.Tensor) -> torch.Tensor:
        """Predict normalized volume from latent.
        
        Args:
            z_q: Quantized latent [B, H, W, D, C] where H=W=D=8, C=4
        
        Returns:
            volume_pred: [B, 1] normalized volume prediction
        """
        # Permute to [B, C, H, W, D] for pooling
        z = z_q.permute(0, 4, 1, 2, 3)
        z = self.pool(z)  # [B, C, 4, 4, 4]
        return self.fc(z)  # [B, 1]

print("VolumeHead defined")
print("  - Predicts normalized volume from latent")
print("  - Regression task with clean gradients")
print("  - Key innovation for v10")

## 5. VQ-VAE v10 Architecture

**Technical:** Same encoder-decoder as v9 (8x8x8 latent), but with added VolumeHead that operates on the quantized latent. Forward pass returns logits, volume prediction, z_q, and indices.

**Simple:** The neural network that compresses Minecraft structures and reconstructs them. v10 adds a special volume prediction head that learns to predict how "full" a structure is.

In [ ]:
class ResidualBlock3D(nn.Module):
    """3D residual block with BatchNorm."""
    def __init__(self, channels: int):
        super().__init__()
        self.conv1 = nn.Conv3d(channels, channels, 3, padding=1)
        self.bn1 = nn.BatchNorm3d(channels)
        self.conv2 = nn.Conv3d(channels, channels, 3, padding=1)
        self.bn2 = nn.BatchNorm3d(channels)

    def forward(self, x):
        residual = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + residual)


class EncoderV10(nn.Module):
    """Encoder: 32x32x32 -> 8x8x8 latent."""
    def __init__(self, in_channels: int = 40, hidden_dim: int = 128, rfsq_dim: int = 4):
        super().__init__()
        self.initial = nn.Sequential(
            nn.Conv3d(in_channels, hidden_dim, 3, padding=1),
            nn.BatchNorm3d(hidden_dim),
            nn.ReLU(inplace=True)
        )
        # 32 -> 16
        self.down1 = nn.Sequential(
            nn.Conv3d(hidden_dim, hidden_dim, 4, stride=2, padding=1),
            nn.BatchNorm3d(hidden_dim),
            nn.ReLU(inplace=True)
        )
        self.res1 = ResidualBlock3D(hidden_dim)
        # 16 -> 8
        self.down2 = nn.Sequential(
            nn.Conv3d(hidden_dim, hidden_dim, 4, stride=2, padding=1),
            nn.BatchNorm3d(hidden_dim),
            nn.ReLU(inplace=True)
        )
        self.res2 = nn.Sequential(*[ResidualBlock3D(hidden_dim) for _ in range(4)])
        self.proj = nn.Conv3d(hidden_dim, rfsq_dim, 3, padding=1)

    def forward(self, x):
        x = self.initial(x)
        x = self.res1(self.down1(x))
        x = self.res2(self.down2(x))
        return self.proj(x)


class DecoderV10(nn.Module):
    """Decoder: 8x8x8 latent -> 32x32x32 output."""
    def __init__(self, rfsq_dim: int = 4, hidden_dim: int = 128, num_blocks: int = 3717):
        super().__init__()
        self.initial = nn.Sequential(
            nn.Conv3d(rfsq_dim, hidden_dim, 3, padding=1),
            nn.BatchNorm3d(hidden_dim),
            nn.ReLU(inplace=True)
        )
        self.res1 = nn.Sequential(*[ResidualBlock3D(hidden_dim) for _ in range(4)])
        # 8 -> 16
        self.up1 = nn.Sequential(
            nn.ConvTranspose3d(hidden_dim, hidden_dim, 4, stride=2, padding=1),
            nn.BatchNorm3d(hidden_dim),
            nn.ReLU(inplace=True)
        )
        self.res2 = ResidualBlock3D(hidden_dim)
        # 16 -> 32
        self.up2 = nn.Sequential(
            nn.ConvTranspose3d(hidden_dim, hidden_dim, 4, stride=2, padding=1),
            nn.BatchNorm3d(hidden_dim),
            nn.ReLU(inplace=True)
        )
        self.final = nn.Conv3d(hidden_dim, num_blocks, 3, padding=1)

    def forward(self, z_q):
        x = self.initial(z_q)
        x = self.res1(x)
        x = self.res2(self.up1(x))
        x = self.up2(x)
        return self.final(x)


class VQVAEv10(nn.Module):
    """VQ-VAE v10 with auxiliary VolumeHead.
    
    Key difference from v9: VolumeHead predicts volume from latent,
    providing clean gradient signal for volume control.
    """
    def __init__(self, vocab_size: int = 3717, emb_dim: int = 40, hidden_dim: int = 128,
                 rfsq_levels: List[int] = None, num_stages: int = 2,
                 pretrained_embeddings: torch.Tensor = None):
        super().__init__()
        if rfsq_levels is None:
            rfsq_levels = [5, 5, 5, 5]
        self.rfsq_dim = len(rfsq_levels)
        self.vocab_size = vocab_size
        
        # Block embeddings
        self.block_emb = nn.Embedding(vocab_size, emb_dim)
        if pretrained_embeddings is not None:
            self.block_emb.weight.data.copy_(pretrained_embeddings)
            self.block_emb.weight.requires_grad = False
        
        # Encoder-Quantizer-Decoder
        self.encoder = EncoderV10(emb_dim, hidden_dim, self.rfsq_dim)
        self.quantizer = RFSQ(rfsq_levels, num_stages)
        self.decoder = DecoderV10(self.rfsq_dim, hidden_dim, vocab_size)
        
        # NEW: Volume prediction head
        self.volume_head = VolumeHead(latent_channels=self.rfsq_dim, hidden_dim=128)

    def forward(self, block_ids):
        """Forward pass with volume prediction.
        
        Returns:
            logits: [B, vocab, 32, 32, 32] block predictions
            volume_pred: [B, 1] normalized volume prediction
            z_q: [B, 8, 8, 8, 4] quantized latent
            indices: list of quantization indices
        """
        z_e = self.encode(block_ids)
        z_q, indices = self.quantize(z_e)
        logits = self.decode(z_q)
        volume_pred = self.volume_head(z_q)  # NEW: predict volume from latent
        return logits, volume_pred, z_q, indices

    def encode(self, block_ids):
        x = self.block_emb(block_ids).permute(0, 4, 1, 2, 3)
        z_e = self.encoder(x).permute(0, 2, 3, 4, 1)
        return z_e

    def quantize(self, z_e):
        return self.quantizer(z_e)

    def decode(self, z_q):
        return self.decoder(z_q.permute(0, 4, 1, 2, 3))

print("VQ-VAE v10 architecture defined")
print("  - 8x8x8 latent (same as v9)")
print("  - NEW: VolumeHead for volume regression")

## 6. V10 Loss Function

**Technical:** Combines focal cross-entropy loss for block prediction with L1 volume regression loss. Uses simple class weights for air tokens instead of air_boost. No frequency weighting.

**Simple:** The error calculation that guides training:
1. **Focal CE**: Focuses on hard-to-predict blocks
2. **Volume L1**: Penalizes wrong volume predictions (regression, not classification)
3. **Air class weight**: Makes air predictions slightly more important (but not aggressive)

### Key Differences from v9
- No air_boost (replaced by class weight)
- No frequency weighting (removed perverse incentive)
- Volume is L1 regression on volume_pred, not L2 on argmax volume

In [ ]:
class V10Loss(nn.Module):
    """Combined focal CE loss + volume regression loss.
    
    Key changes from v9:
    - Volume is L1 regression on volume_pred from VolumeHead
    - No air_boost (replaced by simple class weight)
    - No frequency weighting (removed perverse incentive)
    """
    def __init__(self, air_tokens: Set[int], gamma: float = 2.0,
                 air_class_weight: float = 3.0, volume_weight: float = 10.0):
        super().__init__()
        self.air_tokens = list(air_tokens)
        self.gamma = gamma
        self.air_class_weight = air_class_weight
        self.volume_weight = volume_weight

    def forward(self, logits: torch.Tensor, target: torch.Tensor,
                volume_pred: torch.Tensor, z_q: torch.Tensor) -> Dict[str, torch.Tensor]:
        """Compute combined loss.
        
        Args:
            logits: [B, C, X, Y, Z] block predictions
            target: [B, X, Y, Z] ground truth block IDs
            volume_pred: [B, 1] predicted normalized volume from VolumeHead
            z_q: quantized latent (unused but kept for API consistency)
        """
        device = logits.device
        B, C, X, Y, Z = logits.shape
        total_voxels = X * Y * Z
        
        # Create air mask
        air_tensor = torch.tensor(self.air_tokens, device=device, dtype=target.dtype)
        gt_is_air = torch.isin(target, air_tensor)
        gt_is_struct = ~gt_is_air
        
        # Create class weights (air tokens get higher weight)
        class_weights = torch.ones(C, device=device)
        for t in self.air_tokens:
            if t < C:
                class_weights[t] = self.air_class_weight
        
        # Flatten for loss computation
        logits_flat = logits.permute(0, 2, 3, 4, 1).reshape(-1, C)
        target_flat = target.reshape(-1)
        gt_is_air_flat = gt_is_air.reshape(-1)
        gt_is_struct_flat = gt_is_struct.reshape(-1)
        
        # === Focal Cross-Entropy Loss ===
        log_probs = F.log_softmax(logits_flat, dim=1)
        probs = torch.exp(log_probs)
        p_t = probs.gather(1, target_flat.unsqueeze(1)).squeeze(1)
        focal_weight = (1 - p_t) ** self.gamma
        
        # CE with class weights
        ce_per_sample = F.cross_entropy(logits_flat, target_flat,
                                         weight=class_weights, reduction='none')
        focal_loss = (focal_weight * ce_per_sample).mean()
        
        # === Volume Regression Loss (L1) ===
        # Ground truth normalized volume per sample
        gt_struct_per_sample = gt_is_struct.float().view(B, -1).sum(dim=1)  # [B]
        gt_volume_normalized = gt_struct_per_sample / total_voxels  # [B]
        
        # L1 loss on volume prediction
        volume_loss = F.l1_loss(volume_pred.squeeze(), gt_volume_normalized)
        
        # === Total Loss ===
        total_loss = focal_loss + self.volume_weight * volume_loss
        
        # === Metrics (computed on hard predictions) ===
        with torch.no_grad():
            pred = logits_flat.argmax(dim=1)
            pred_is_air = torch.isin(pred, air_tensor)
            pred_is_struct = ~pred_is_air
            
            # Overall accuracy
            correct = (pred == target_flat)
            accuracy = correct.float().mean()
            
            # Building accuracy (correct predictions at GT structure locations)
            building_acc = correct[gt_is_struct_flat].float().mean() if gt_is_struct_flat.any() else torch.tensor(0.0)
            
            # Air accuracy
            air_acc = correct[gt_is_air_flat].float().mean() if gt_is_air_flat.any() else torch.tensor(0.0)
            
            # Recall: % of GT structure predicted as structure
            recall = pred_is_struct[gt_is_struct_flat].float().mean() if gt_is_struct_flat.any() else torch.tensor(0.0)
            
            # Precision: % of predicted structure that is correct
            precision = correct[pred_is_struct & gt_is_struct_flat].sum().float() / (pred_is_struct.sum().float() + 1e-6)
            
            # False air rate: % of GT structure predicted as air
            false_air_rate = pred_is_air[gt_is_struct_flat].float().mean() if gt_is_struct_flat.any() else torch.tensor(0.0)
            
            # Volume ratio (hard predictions)
            pred_volume = pred_is_struct.float().sum()
            gt_volume = gt_is_struct_flat.float().sum()
            volume_ratio = pred_volume / (gt_volume + 1e-6)
        
        return {
            'loss': total_loss,
            'focal_loss': focal_loss.detach(),
            'volume_loss': volume_loss.detach(),
            'volume_pred': volume_pred.mean().detach(),
            'volume_gt': gt_volume_normalized.mean().detach(),
            'volume_ratio': volume_ratio,
            'accuracy': accuracy,
            'building_acc': building_acc,
            'air_acc': air_acc,
            'recall': recall,
            'precision': precision,
            'false_air_rate': false_air_rate,
        }

print("V10Loss defined")
print(f"  - Focal gamma: 2.0")
print(f"  - Air class weight: 3.0x (simple, not boost)")
print(f"  - Volume L1 weight: 10.0x (regression loss)")
print(f"  - NO frequency weighting")
print(f"  - NO air_boost")

## 7. Configuration

**Technical:** Hyperparameters for v10 training. Key changes:
- Single-phase training (no two-phase needed with VolumeHead)
- No frequency weighting
- Simple air class weight instead of air_boost
- Volume head weight for regression loss

**Simple:** Training settings. The VolumeHead provides clean gradient signal, so we don't need the two-phase approach from v9.

In [ ]:
# === Data Paths ===
DATA_DIR = f"{DRIVE_BASE}/splits/train"
VAL_DIR = f"{DRIVE_BASE}/splits/val"
VOCAB_PATH = f"{DRIVE_BASE}/vocabulary/tok2block.json"
V3_EMBEDDINGS_PATH = f"{DRIVE_BASE}/embeddings/block_embeddings_v3.npy"

# Verify paths
print("Checking paths...")
for name, path in [('DATA_DIR', DATA_DIR), ('VAL_DIR', VAL_DIR),
                   ('VOCAB_PATH', VOCAB_PATH), ('V3_EMBEDDINGS_PATH', V3_EMBEDDINGS_PATH)]:
    exists = Path(path).exists()
    print(f"  {name}: {'[OK]' if exists else '[NOT FOUND]'}")

# === v10 Architecture ===
HIDDEN_DIM = 128
RFSQ_LEVELS = [5, 5, 5, 5]  # 4 dims x 5 levels
NUM_STAGES = 2

# === v10 Loss Weights ===
FOCAL_GAMMA = 2.0
AIR_CLASS_WEIGHT = 3.0     # Simple class weight (NOT air_boost)
VOLUME_HEAD_WEIGHT = 10.0  # L1 loss on volume prediction
# NO frequency weighting (removed perverse incentive)
# NO air_boost (replaced by class weight)

# === Training (Single Phase) ===
EPOCHS = 20               # Single phase, no two-phase needed
BATCH_SIZE = 4
BASE_LR = 2e-4
USE_AMP = True
GRAD_ACCUM_STEPS = 4
SEED = 42
NUM_WORKERS = 2

print(f"\n{'='*60}")
print("V10 CONFIGURATION - VOLUME HEAD")
print(f"{'='*60}")
print(f"Key innovation: VolumeHead predicts volume from latent")
print(f"\nArchitecture:")
print(f"  - Latent: 8x8x8")
print(f"  - VolumeHead: MLP on pooled latent")
print(f"\nLoss:")
print(f"  - Focal gamma: {FOCAL_GAMMA}")
print(f"  - Air class weight: {AIR_CLASS_WEIGHT}x")
print(f"  - Volume head weight: {VOLUME_HEAD_WEIGHT}x")
print(f"  - NO frequency weighting")
print(f"  - NO air_boost")
print(f"\nTraining:")
print(f"  - Epochs: {EPOCHS} (single phase)")
print(f"  - Batch size: {BATCH_SIZE}")
print(f"  - Effective batch: {BATCH_SIZE * GRAD_ACCUM_STEPS}")
print(f"{'='*60}")

## 8. Load Vocabulary and Embeddings

**Technical:** Loads block vocabulary (token to block name) and pre-trained block embeddings. Identifies air tokens.

**Simple:** Loads the dictionary mapping numbers to Minecraft block names.

In [ ]:
with open(VOCAB_PATH, 'r') as f:
    tok2block = {int(k): v for k, v in json.load(f).items()}

VOCAB_SIZE = len(tok2block)
print(f"Vocabulary size: {VOCAB_SIZE}")

# Find air tokens
AIR_TOKENS: Set[int] = set()
for tok, block in tok2block.items():
    if 'air' in block.lower() and 'stair' not in block.lower():
        AIR_TOKENS.add(tok)
        print(f"  Air token: {tok} = {block}")

AIR_TOKENS_TENSOR = torch.tensor(sorted(AIR_TOKENS), dtype=torch.long)

# Load embeddings
v3_embeddings = np.load(V3_EMBEDDINGS_PATH).astype(np.float32)
EMBEDDING_DIM = v3_embeddings.shape[1]
print(f"\nEmbeddings: {v3_embeddings.shape}")

## 9. Dataset

**Technical:** PyTorch Dataset that loads Minecraft structures from HDF5 files.

**Simple:** Defines how to load structure data for training.

In [ ]:
class VQVAEDataset(Dataset):
    def __init__(self, data_dir: str):
        self.h5_files = sorted(Path(data_dir).glob("*.h5"))
        if not self.h5_files:
            raise ValueError(f"No H5 files in {data_dir}")
        print(f"Found {len(self.h5_files)} structures")

    def __len__(self):
        return len(self.h5_files)

    def __getitem__(self, idx):
        with h5py.File(self.h5_files[idx], 'r') as f:
            structure = f[list(f.keys())[0]][:].astype(np.int64)
        return torch.from_numpy(structure).long()

train_dataset = VQVAEDataset(DATA_DIR)
val_dataset = VQVAEDataset(VAL_DIR)

## 10. Create Model, Criterion, Optimizer

**Technical:** Instantiates VQ-VAE v10 with VolumeHead, V10Loss criterion, AdamW optimizer with cosine annealing.

**Simple:** Creates the neural network and training setup.

In [ ]:
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

model = VQVAEv10(
    vocab_size=VOCAB_SIZE,
    emb_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    rfsq_levels=RFSQ_LEVELS,
    num_stages=NUM_STAGES,
    pretrained_embeddings=torch.from_numpy(v3_embeddings),
).to(device)

total_params = sum(p.numel() for p in model.parameters())
volume_head_params = sum(p.numel() for p in model.volume_head.parameters())
print(f"Total parameters: {total_params:,}")
print(f"VolumeHead parameters: {volume_head_params:,}")

criterion = V10Loss(
    air_tokens=AIR_TOKENS,
    gamma=FOCAL_GAMMA,
    air_class_weight=AIR_CLASS_WEIGHT,
    volume_weight=VOLUME_HEAD_WEIGHT,
).to(device)

optimizer = optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)

print("\nModel, criterion, optimizer created")
print(f"  VolumeHead added to model")

## 11. Data Loaders

**Technical:** Creates PyTorch DataLoaders for batching and parallel loading.

**Simple:** Sets up efficient data loading.

In [ ]:
g = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, generator=g)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM_STEPS}")

## 12. Training Functions

**Technical:** train_epoch() and validate() functions for single-phase training. Tracks all metrics including volume prediction accuracy.

**Simple:** Core training logic that shows structures to the model, computes errors, and adjusts parameters.

In [ ]:
def train_epoch(model, criterion, loader, optimizer, scaler, device):
    """Train one epoch."""
    model.train()
    model.quantizer.reset_usage()
    
    metrics_sum = defaultdict(float)
    n = 0
    optimizer.zero_grad()
    
    for batch_idx, batch in enumerate(tqdm(loader, desc="Train", leave=False)):
        batch = batch.to(device)
        
        with torch.amp.autocast('cuda', enabled=USE_AMP):
            logits, volume_pred, z_q, indices = model(batch)
            loss_dict = criterion(logits, batch, volume_pred, z_q)
            loss = loss_dict['loss'] / GRAD_ACCUM_STEPS
        
        scaler.scale(loss).backward()
        
        if (batch_idx + 1) % GRAD_ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        
        with torch.no_grad():
            for k, v in loss_dict.items():
                metrics_sum[k] += v.item() if torch.is_tensor(v) else v
        n += 1
    
    metrics = {k: v / n for k, v in metrics_sum.items()}
    
    # RFSQ usage stats
    for name, (usage, perp) in model.quantizer.get_usage_stats().items():
        metrics[f'{name}_usage'] = usage
        metrics[f'{name}_perplexity'] = perp
    
    return metrics


@torch.no_grad()
def validate(model, criterion, loader, device):
    """Validate."""
    model.eval()
    model.quantizer.reset_usage()
    
    metrics_sum = defaultdict(float)
    n = 0
    
    for batch in tqdm(loader, desc="Val", leave=False):
        batch = batch.to(device)
        
        with torch.amp.autocast('cuda', enabled=USE_AMP):
            logits, volume_pred, z_q, indices = model(batch)
            loss_dict = criterion(logits, batch, volume_pred, z_q)
        
        for k, v in loss_dict.items():
            metrics_sum[k] += v.item() if torch.is_tensor(v) else v
        n += 1
    
    metrics = {k: v / n for k, v in metrics_sum.items()}
    
    for name, (usage, perp) in model.quantizer.get_usage_stats().items():
        metrics[f'{name}_usage'] = usage
        metrics[f'{name}_perplexity'] = perp
    
    return metrics

print("Training functions defined")

## 13. Training Loop

**Technical:** Single-phase training loop. Tracks volume_pred accuracy alongside standard metrics.

**Simple:** The main training process - runs for 20 epochs, tracking all important metrics.

In [ ]:
print("="*70)
print("VQ-VAE V10 TRAINING - AUXILIARY VOLUME HEAD")
print("="*70)
print(f"Key innovation: VolumeHead predicts volume from latent")
print(f"Epochs: {EPOCHS} (single phase)")
print(f"Target: volume_ratio 0.9-1.1x, recall >85%, FAR <10%")
print("="*70)

history = {
    'train_loss': [], 'val_loss': [],
    'train_focal_loss': [], 'val_focal_loss': [],
    'train_volume_loss': [], 'val_volume_loss': [],
    'train_volume_pred': [], 'val_volume_pred': [],
    'train_volume_gt': [], 'val_volume_gt': [],
    'train_volume_ratio': [], 'val_volume_ratio': [],
    'train_recall': [], 'val_recall': [],
    'train_precision': [], 'val_precision': [],
    'train_accuracy': [], 'val_accuracy': [],
    'train_building_acc': [], 'val_building_acc': [],
    'train_air_acc': [], 'val_air_acc': [],
    'train_false_air_rate': [], 'val_false_air_rate': [],
    'train_stage0_perplexity': [], 'val_stage0_perplexity': [],
    'learning_rate': [],
}

best_metric = 0
best_epoch = 0
start_time = time.time()

for epoch in range(EPOCHS):
    train_m = train_epoch(model, criterion, train_loader, optimizer, scaler, device)
    val_m = validate(model, criterion, val_loader, device)
    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]
    
    # Record metrics
    for prefix, m in [('train', train_m), ('val', val_m)]:
        history[f'{prefix}_loss'].append(m.get('loss', 0))
        history[f'{prefix}_focal_loss'].append(m.get('focal_loss', 0))
        history[f'{prefix}_volume_loss'].append(m.get('volume_loss', 0))
        history[f'{prefix}_volume_pred'].append(m.get('volume_pred', 0))
        history[f'{prefix}_volume_gt'].append(m.get('volume_gt', 0))
        history[f'{prefix}_volume_ratio'].append(m.get('volume_ratio', 0))
        history[f'{prefix}_recall'].append(m.get('recall', 0))
        history[f'{prefix}_precision'].append(m.get('precision', 0))
        history[f'{prefix}_accuracy'].append(m.get('accuracy', 0))
        history[f'{prefix}_building_acc'].append(m.get('building_acc', 0))
        history[f'{prefix}_air_acc'].append(m.get('air_acc', 0))
        history[f'{prefix}_false_air_rate'].append(m.get('false_air_rate', 0))
        history[f'{prefix}_stage0_perplexity'].append(m.get('stage0_perplexity', 0))
    history['learning_rate'].append(current_lr)
    
    # Best model: balance volume ratio and accuracy
    vol_score = 1.0 - abs(val_m['volume_ratio'] - 1.0)
    acc_score = val_m.get('building_acc', val_m.get('accuracy', 0))
    current_metric = 0.5 * vol_score + 0.5 * acc_score
    
    if current_metric > best_metric:
        best_metric = current_metric
        best_epoch = epoch + 1
        torch.save(model.state_dict(), f"{OUTPUT_DIR}/vqvae_v10_best.pt")
    
    # Checkpoint every 5 epochs
    if (epoch + 1) % 5 == 0:
        torch.save(model.state_dict(), f"{OUTPUT_DIR}/vqvae_v10_epoch{epoch+1}.pt")
    
    # Print progress
    vol_str = f"{val_m['volume_ratio']:.3f}x"
    vol_ok = 0.9 <= val_m['volume_ratio'] <= 1.1
    vol_pred_str = f"{val_m['volume_pred']:.3f}"
    vol_gt_str = f"{val_m['volume_gt']:.3f}"
    
    print(f"E{epoch+1:2d} | Build: {val_m['building_acc']:.1%} | "
          f"Vol: {vol_str} {'OK' if vol_ok else '!!'} | "
          f"Pred: {vol_pred_str} vs GT: {vol_gt_str} | "
          f"Recall: {val_m['recall']:.1%} | FAR: {val_m['false_air_rate']:.1%}")

train_time = time.time() - start_time
print(f"\n{'='*70}")
print("TRAINING COMPLETE")
print(f"{'='*70}")
print(f"Time: {train_time/60:.1f} minutes")
print(f"Best epoch: {best_epoch}")
print(f"Final volume ratio: {history['val_volume_ratio'][-1]:.3f}x")
print(f"Final recall: {history['val_recall'][-1]:.1%}")
print(f"Final false air rate: {history['val_false_air_rate'][-1]:.1%}")
print(f"Final building accuracy: {history['val_building_acc'][-1]:.1%}")

## 14. Plot Training Results

**Technical:** Creates plots showing all metrics including volume prediction accuracy.

**Simple:** Visualizes training progress - we want volume ratio ~1.0 and low false air rate.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 14))
epochs_range = range(1, len(history['train_loss']) + 1)

# 1. Volume Ratio (KEY METRIC)
ax = axes[0, 0]
ax.plot(epochs_range, history['train_volume_ratio'], 'b-', label='Train', linewidth=2)
ax.plot(epochs_range, history['val_volume_ratio'], 'r--', label='Val', linewidth=2)
ax.axhline(y=1.0, color='g', linestyle='--', linewidth=2, label='Target (1.0x)')
ax.axhline(y=0.62, color='orange', linestyle=':', alpha=0.7, label='v9 result (0.62x)')
ax.fill_between(epochs_range, 0.9, 1.1, alpha=0.2, color='green', label='Target zone')
ax.set_xlabel('Epoch'); ax.set_ylabel('Volume Ratio')
ax.set_title('Volume Ratio (KEY METRIC)', fontweight='bold', fontsize=12, color='darkred')
ax.legend(loc='upper right', fontsize=8); ax.grid(True, alpha=0.3)
ax.set_ylim(0, max(2.0, max(history['val_volume_ratio']) * 1.1))

# 2. Volume Prediction vs GT (NEW in v10)
ax = axes[0, 1]
ax.plot(epochs_range, history['train_volume_pred'], 'b-', label='Train Pred', linewidth=2)
ax.plot(epochs_range, history['val_volume_pred'], 'r--', label='Val Pred', linewidth=2)
ax.plot(epochs_range, history['val_volume_gt'], 'g:', label='Val GT', linewidth=2)
ax.set_xlabel('Epoch'); ax.set_ylabel('Normalized Volume')
ax.set_title('VolumeHead Prediction vs GT', fontweight='bold', fontsize=12)
ax.legend(loc='upper right'); ax.grid(True, alpha=0.3)

# 3. False Air Rate (KEY for v10)
ax = axes[0, 2]
ax.plot(epochs_range, history['train_false_air_rate'], 'b-', label='Train', linewidth=2)
ax.plot(epochs_range, history['val_false_air_rate'], 'r--', label='Val', linewidth=2)
ax.axhline(y=0.10, color='g', linestyle='--', alpha=0.5, label='Target (<10%)')
ax.axhline(y=0.40, color='orange', linestyle=':', alpha=0.7, label='v9 result (40%)')
ax.set_xlabel('Epoch'); ax.set_ylabel('Rate')
ax.set_title('False Air Rate (Erasure)', fontweight='bold', fontsize=12, color='darkred')
ax.legend(loc='upper right'); ax.grid(True, alpha=0.3)

# 4. Recall
ax = axes[1, 0]
ax.plot(epochs_range, history['train_recall'], 'b-', label='Train', linewidth=2)
ax.plot(epochs_range, history['val_recall'], 'r--', label='Val', linewidth=2)
ax.axhline(y=0.85, color='g', linestyle='--', alpha=0.5, label='Target (85%)')
ax.set_xlabel('Epoch'); ax.set_ylabel('Recall')
ax.set_title('Recall (Structure Preservation)', fontweight='bold', fontsize=12)
ax.legend(loc='lower right'); ax.grid(True, alpha=0.3); ax.set_ylim(0, 1.05)

# 5. Building Accuracy
ax = axes[1, 1]
ax.plot(epochs_range, history['train_building_acc'], 'b-', label='Train', linewidth=2)
ax.plot(epochs_range, history['val_building_acc'], 'r--', label='Val', linewidth=2)
ax.axhline(y=0.492, color='g', linestyle=':', alpha=0.5, label='v6-freq (49.2%)')
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
ax.set_title('Building Accuracy', fontweight='bold', fontsize=12)
ax.legend(loc='lower right'); ax.grid(True, alpha=0.3)

# 6. Precision
ax = axes[1, 2]
ax.plot(epochs_range, history['train_precision'], 'b-', label='Train', linewidth=2)
ax.plot(epochs_range, history['val_precision'], 'r--', label='Val', linewidth=2)
ax.set_xlabel('Epoch'); ax.set_ylabel('Precision')
ax.set_title('Precision', fontweight='bold', fontsize=12)
ax.legend(loc='lower right'); ax.grid(True, alpha=0.3); ax.set_ylim(0, 1.05)

# 7. Loss Components
ax = axes[2, 0]
ax.plot(epochs_range, history['val_focal_loss'], 'b-', label='Focal CE', linewidth=2)
ax.plot(epochs_range, history['val_volume_loss'], 'r-', label='Volume L1', linewidth=2)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Loss Components (Val)', fontweight='bold', fontsize=12)
ax.legend(); ax.grid(True, alpha=0.3)

# 8. Total Loss
ax = axes[2, 1]
ax.plot(epochs_range, history['train_loss'], 'b-', label='Train', linewidth=2)
ax.plot(epochs_range, history['val_loss'], 'r--', label='Val', linewidth=2)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Total Loss', fontweight='bold', fontsize=12)
ax.legend(); ax.grid(True, alpha=0.3)

# 9. RFSQ Perplexity
ax = axes[2, 2]
ax.plot(epochs_range, history['train_stage0_perplexity'], 'b-', label='Train', linewidth=2)
ax.plot(epochs_range, history['val_stage0_perplexity'], 'r--', label='Val', linewidth=2)
ax.set_xlabel('Epoch'); ax.set_ylabel('Perplexity')
ax.set_title('RFSQ Perplexity', fontweight='bold', fontsize=12)
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/vqvae_v10_training.png", dpi=150, bbox_inches='tight')
plt.show()

# Summary
print("\n" + "="*70)
print("V10 RESULTS SUMMARY")
print("="*70)
final_vol = history['val_volume_ratio'][-1]
final_recall = history['val_recall'][-1]
final_build = history['val_building_acc'][-1]
final_far = history['val_false_air_rate'][-1]

print(f"Final volume ratio: {final_vol:.3f}x")
print(f"Final recall: {final_recall:.1%}")
print(f"Final false air rate: {final_far:.1%}")
print(f"Final building accuracy: {final_build:.1%}")
print()

# Success criteria
vol_ok = 0.9 <= final_vol <= 1.1
recall_ok = final_recall >= 0.85
far_ok = final_far < 0.10
acc_ok = final_build >= 0.45

print("Success Criteria:")
print(f"  Volume 0.9-1.1x: {'PASS' if vol_ok else 'FAIL'} ({final_vol:.3f}x)")
print(f"  Recall >= 85%: {'PASS' if recall_ok else 'FAIL'} ({final_recall:.1%})")
print(f"  FAR < 10%: {'PASS' if far_ok else 'FAIL'} ({final_far:.1%})")
print(f"  Build acc >= 45%: {'PASS' if acc_ok else 'FAIL'} ({final_build:.1%})")
print()

if vol_ok and recall_ok and far_ok and acc_ok:
    print("SUCCESS! All v10 targets met.")
elif vol_ok and far_ok:
    print("GOOD PROGRESS: Volume and false air fixed! May need more epochs for accuracy.")
elif vol_ok:
    print("PARTIAL: Volume fixed but still erasing structures.")
else:
    print("NEEDS WORK: Check VolumeHead loss weights.")

## 15. Save Results

**Technical:** Serializes training configuration, final metrics, and history to JSON. Saves final checkpoint.

**Simple:** Saves all results and the trained model.

In [ ]:
results = {
    'config': {
        'version': 'v10-VOLUME-HEAD',
        'latent_resolution': '8x8x8',
        'hidden_dim': HIDDEN_DIM,
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'base_lr': BASE_LR,
        'focal_gamma': FOCAL_GAMMA,
        'air_class_weight': AIR_CLASS_WEIGHT,
        'volume_head_weight': VOLUME_HEAD_WEIGHT,
        'seed': SEED,
    },
    'results': {
        'final_volume_ratio': float(history['val_volume_ratio'][-1]),
        'final_recall': float(history['val_recall'][-1]),
        'final_precision': float(history['val_precision'][-1]),
        'final_building_acc': float(history['val_building_acc'][-1]),
        'final_air_acc': float(history['val_air_acc'][-1]),
        'final_false_air_rate': float(history['val_false_air_rate'][-1]),
        'final_volume_pred': float(history['val_volume_pred'][-1]),
        'final_volume_gt': float(history['val_volume_gt'][-1]),
        'best_epoch': best_epoch,
        'training_time_min': float(train_time / 60),
        'volume_target_met': bool(0.9 <= history['val_volume_ratio'][-1] <= 1.1),
        'recall_target_met': bool(history['val_recall'][-1] >= 0.85),
        'far_target_met': bool(history['val_false_air_rate'][-1] < 0.10),
        'accuracy_target_met': bool(history['val_building_acc'][-1] >= 0.45),
    },
    'history': {k: [float(x) for x in v] for k, v in history.items()},
}

with open(f"{OUTPUT_DIR}/vqvae_v10_results.json", 'w') as f:
    json.dump(results, f, indent=2)

torch.save(model.state_dict(), f"{OUTPUT_DIR}/vqvae_v10_final.pt")

print("Results saved:")
print(f"  {OUTPUT_DIR}/vqvae_v10_results.json")
print(f"  {OUTPUT_DIR}/vqvae_v10_best.pt")
print(f"  {OUTPUT_DIR}/vqvae_v10_final.pt")
print(f"  {OUTPUT_DIR}/vqvae_v10_training.png")
print("\nDone!")